# 35. CatBoost's own ordered target statistics

**One variable against ledger row 26** (`catboost_te`, CV 0.966915): **who builds the
target statistics.** Row 26 hands CatBoost the 36 features our nested encoder produced.
This hands CatBoost the 12 raw columns as string levels and lets it compute its own
ordered target statistics internally. Same learner, same folds, same seed, the same
`learning_rate * n_estimators = 100` budget convention.

Everything else is held: `SEED = 42`, the five frozen folds (sha `ec282b0968059676`),
`depth=6`, `lr=0.05`, `n_estimators=2000`, no early stopping.

## Where this came from, and why its CV scheme was checked first

The workspace rules forbid copying a public notebook's approach without reading its CV
scheme, so that was done before anything here was written.

Two public notebooks report this result. tomasa2's ablation
(`s6e8-what-moved-the-score-and-what-didn-t`, §6.1) measures **+0.0004 over their own
hand-rolled nested encoder** and calls it their best single model. srcJ's
`s6e8-diversity-beats-strength` lists it as the top member of a 177-model pool, solo OOF
**0.968601**, and the **largest stack coefficient in their table at +0.749**.

Both run `StratifiedKFold(5, shuffle=True, random_state=42)` over `train.csv` in original
file row order. That is **bit-identical to this repo's split**, verified by regenerating
the partition and matching the recorded sha rather than by reading their code and
believing it. So their numbers are directly comparable to ours, which is unusual and is
the reason this idea is worth acting on rather than merely noting.

## Why this is not the reopening the rule says to skip

The six-point rule` is three for three on learners and **nought for four on
hyperparameters**, and the last four reopenings this repo ran were all knobs and all null.
`max_ctr_complexity` below is a knob and is treated as one.

The arm that matters is not a knob. It replaces our encoder with a different estimator of
the same quantity: ours applies one global smoothing constant to every level, CatBoost's
permutation scheme shrinks per row and per position. This repo records the smoothing
constant as **inert** (rows 46 to 49) and the inner split count as **inert** (rows 50 to
54), which is the evidence that our encoder had stopped being improvable *within its own
parameterisation*. A different estimator is not a value of one of those knobs.

## The prediction, written before the run

- **Arm A (`max_ctr_complexity=1`), the clean one-variable comparison.** Predict solo CV
  **0.9675 to 0.9690**, so above row 26 (0.966915) and above row 38 (0.967099, the best
  single model in this repo).
- **Arm B (`max_ctr_complexity=2`)** adds CatBoost's own pairwise category combinations.
  Predict **null or negative against arm A**. Rows 55 to 57 found explicit 2-way target
  encoding worth nothing, and rows 62 to 65 found extra tree depth actively costly. If B
  also comes back flat, that is the same finding through a third, independent mechanism.
- **Stack coefficient:** predict a large positive one, because this is the first member in
  the repo that does not consume our encoder. Membership is decided in `36`, not here.

**Track record, stated so this reads honestly either way.** The last three pre-registered
predictions in this repo were wrong, half wrong, and wrong (rows 32, 33, 34), while the
rule itself called seven of seven. The rule assigns a *representation* change a high prior
and that is the basis of arm A's prediction, not the mechanism story above it.

## What this notebook decides, and what it does not

The single-model CV is description. This repo established on 2026-08-12, expensively, that
a member's own CV is close to irrelevant to whether a fitted combiner wants it: the neural
model sits 0.0247 behind and carries a +0.1178 coefficient. **The membership decision is
`36_stack_native.ipynb` and it is a separate ledger row**, because it changes a different
variable. No submission csv is written here.

In [ ]:
# One flag. The run always goes top to bottom on Kaggle.
SMOKE = True

SEED = 42

# Row 26's budget convention, held: learning_rate * n_estimators = 100. Kept so that
# this is one variable against row 26 and comparable to rows 17 and 38 as well.
LR = 0.05
N_EST = 2000
DEPTH = 6
BENCH_EST = 200
PROBE_FOLD = 0

# NO early stopping. Public notebooks on this competition early-stop on the validation
# fold, which is the fold their OOF is scored on. That is a mild optimism and, more to
# the point here, it would stop this being one variable against rows 26 and 38, which
# use a fixed budget. Cost: this arm is handicapped against the public numbers quoted
# in the header. That is the right trade for a comparison against our own ledger.

# The two arms. complexity 1 is 1-way ordered target statistics, the direct analogue of
# our encoder. complexity 2 lets CatBoost build pairwise category combinations.
ARMS = [("cat_native_c1", 1), ("cat_native_c2", 2)]

# Row 26 ran on Kaggle at the library default thread count. Matching it is part of
# matching it. Measured 2026-08-19 and written up: at 16,000 rows this
# machine runs 101x slower at all-threads than at one thread, monotone in the count.
# A smoke run produces no ledger number, so overriding it there costs nothing.
THREADS = -1

# Hard guard. Kaggle allows 12h on CPU; the bench stage projects the full run and stops
# here rather than after five hours of a run that was never going to finish.
MAX_HOURS = 9.0

# Row 26: this learner, these folds, this seed, our encoder.
BASELINE_NAME = "catboost_te"
BASELINE_CV = 0.966915
# Row 38: the best single model in the repo, for context only.
XGB_TE_CV = 0.967099
EXPECTED_FOLD_SHA = "ec282b0968059676"

print(f"SMOKE = {SMOKE}   lr {LR}   n_estimators {N_EST}   depth {DEPTH}")
print(f"arms: {[a for a, _ in ARMS]}")

## Stage 1. Data, folds, leak checklist

The fold checksum is the only thing standing between an out-of-fold vector that blends and
one that is silently misaligned, so it is checked before anything trains rather than after.
Lifted from `34_xgb_depth.ipynb`.

In [ ]:
import gc
import hashlib
import time
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

KAG = Path("/kaggle/input")
ON_KAGGLE = KAG.exists()
LOCAL = next((b for b in [Path.cwd(), *Path.cwd().parents]
              if (b / "data" / "raw" / "train.csv").exists()), None)


def locate(name):
    if ON_KAGGLE:
        hits = sorted(KAG.rglob(name))
        if hits:
            return hits[0]
    if LOCAL is not None:
        for d in ("data/raw", "artifacts/oof", "notebooks", "submissions"):
            p = LOCAL / d / name
            if p.exists():
                return p
    raise FileNotFoundError(name)


OUT = Path("/kaggle/working") if ON_KAGGLE else LOCAL / "artifacts" / "oof"
print(f"running {'on Kaggle' if ON_KAGGLE else 'locally'}, writing to {OUT}")

train_full = pd.read_csv(locate("train.csv"))
test = pd.read_csv(locate("test.csv"))
TARGET = "addicted_label"
COLS = [c for c in train_full.columns if c not in ("id", TARGET)]

checks = {
    "id is not a feature": "id" not in COLS,
    "target is not a feature": TARGET not in COLS,
    "train and test ids do not overlap":
        not (set(train_full["id"]) & set(test["id"])),
    "train and test feature lists match":
        COLS == [c for c in test.columns if c != "id"],
}
for name, ok in checks.items():
    print(f"  [{'ok' if ok else 'FAIL'}] {name}")
LEAK_OK = all(checks.values())

if SMOKE:
    ROW_IDX = np.sort(train_full.sample(20000, random_state=0).index.to_numpy())
    train = train_full.loc[ROW_IDX].reset_index(drop=True)
    test = test.head(5000).reset_index(drop=True)
    N_EST, BENCH_EST = 200, 50
    THREADS = 1
else:
    ROW_IDX = np.arange(len(train_full))
    train = train_full

y = train[TARGET].to_numpy()
folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=SEED).split(train, y)):
    folds[va] = i

sha = hashlib.sha256(folds.tobytes()).hexdigest()[:16]
ALIGNED = sha == EXPECTED_FOLD_SHA
print()
print(f"rows {len(train):,}   target rate {y.mean():.6f}")
print(f"fold sizes {np.bincount(folds).tolist()}")
print(f"fold sha {sha}  expected {EXPECTED_FOLD_SHA}")
if SMOKE:
    print("SMOKE: subsampled, so the sha is EXPECTED to differ. Not a check.")
else:
    print("fold alignment: VERIFIED" if ALIGNED else
          "fold alignment: MISMATCH - the OOF from this run is not blendable")

## Stage 2. The representation

Every one of the 12 columns becomes a string level and is handed to CatBoost as a
categorical. That is the whole feature set. No encoder of ours runs at all.

**Missing values get their own explicit level.** The obvious `df[c].astype(str)` writes
the literal `"nan"` on pandas 2.x and is fine, but on pandas 3.0 the new string dtype
preserves NA instead, and every missing row then drops silently out of the level
statistics. tomasa2's notebook documents this trap and measures it; it produces no error
and no warning. `astype(object).fillna(...).astype(str)` is correct on both, and the
coverage assertion below is what turns it from a belief into a check.

**Why casting continuous columns to categorical is not as reckless as it sounds.** The
counts are printed below. `daily_screen_time_hours` has about 1,389 distinct values over
691,369 rows, roughly 500 rows per level, so a target statistic at each of 1,389 points is
a well estimated quantity. This repo already records the mechanism from the other side:
`notifications_per_day` and `app_opens_per_day` behave as lookup keys rather than
quantities, with adjacent integers carrying very different target rates. This is the same
observation the 1-way target encoding in row 17 was exploiting.

In [ ]:
MISSING = "__missing__"


def levels(df):
    """String levels for every column, missing as an explicit level.

    Routed through object so this behaves identically on pandas 2.x and 3.0.
    """
    return pd.DataFrame({c: df[c].astype(object).fillna(MISSING).astype(str).values
                         for c in COLS})


L_TR = levels(train)
L_TE = levels(test)

# The trap, asserted rather than trusted: every row must be covered by the level table.
for c in COLS:
    n = pd.DataFrame({"lv": L_TR[c].values, "y": y}).groupby("lv")["y"].size().sum()
    assert n == len(train), f"{c}: level table covers {n:,} of {len(train):,} rows"
assert not L_TR.isna().any().any()
assert not L_TE.isna().any().any()
print(f"pandas {pd.__version__}: level tables cover all {len(train):,} train rows")
print()

print(f"{'column':<26s}{'levels':>9s}{'rows/level':>12s}")
for c in COLS:
    k = L_TR[c].nunique()
    print(f"{c:<26s}{k:>9,d}{len(train) / k:>12,.0f}")

CAT_IDX = list(range(len(COLS)))
print(f"\n{len(COLS)} columns, all {len(CAT_IDX)} handed to CatBoost as cat_features")

### The leak checks

The first two checks in `13_target_encoding.ipynb` ask whether a row's own target reached
its own encoding. **There is no encoding of ours here to check**, which removes that class
of leak by construction rather than by verification, and that is worth saying plainly
rather than quietly dropping the stage.

What replaces it is the check that matters for this representation: the levels are built
from features only, so the level tables must be **bit-identical under any permutation of
the target**. If they are not, something target-derived has crept into the feature build.
CatBoost's own ordered target statistics are computed inside `fit` on the rows it is given,
and it is never given a validation or test label, so the fold loop below is what bounds
them.

The third check from `13` is kept in spirit and inverted: the *model* must still move when
training targets change, otherwise this cell would pass while training on noise.

In [ ]:
rng = np.random.default_rng(0)
y_perm = rng.permutation(y)

L_perm = levels(train)
same = all((L_perm[c].values == L_TR[c].values).all() for c in COLS)
print(f"1. level tables are a pure function of the features:      {same}")

# The features never see y at all, so this is stronger than "about zero": it is exact.
built_from_y = False
print(f"2. any target-derived column in the feature set:          {built_from_y}")

# 3. A model trained on permuted targets must collapse to chance on held-out rows.
_tr = np.where(folds != 0)[0][:20000]
_va = np.where(folds == 0)[0][:20000]
m = CatBoostClassifier(iterations=50, learning_rate=LR, depth=DEPTH, random_seed=SEED,
                       max_ctr_complexity=1, verbose=0, allow_writing_files=False,
                       thread_count=THREADS)
m.fit(Pool(L_TR.iloc[_tr], y_perm[_tr], cat_features=CAT_IDX))
auc_perm = roc_auc_score(y[_va], m.predict_proba(Pool(L_TR.iloc[_va],
                                                      cat_features=CAT_IDX))[:, 1])
m.fit(Pool(L_TR.iloc[_tr], y[_tr], cat_features=CAT_IDX))
auc_real = roc_auc_score(y[_va], m.predict_proba(Pool(L_TR.iloc[_va],
                                                      cat_features=CAT_IDX))[:, 1])
print(f"3. trained on PERMUTED targets, held-out AUC: {auc_perm:.6f}  (want ~0.5)")
print(f"   trained on real targets,     held-out AUC: {auc_real:.6f}  (want >> 0.5)")

CLEAN = same and not built_from_y and abs(auc_perm - 0.5) < 0.02 and auc_real > 0.9
print()
print(f"leak checks: {'PASS' if CLEAN else 'FAILED'}")
del m
gc.collect()

## Stage 3. Bench, determinism, and the projection

Two things before the full run. The same fold is trained twice at a short budget and the
scores must be bit-identical, because this repo records that a non-deterministic learner
makes every paired comparison downstream unreadable. Then the full run is projected from
the bench, and the notebook **stops here** if it will not fit inside `MAX_HOURS`, rather
than discovering that five hours in.

In [ ]:
def hhmm(s):
    s = int(s)
    return f"{s // 3600}h {s % 3600 // 60:02d}m" if s >= 3600 else f"{s // 60}m {s % 60:02d}s"


def note(msg):
    print(f"{time.strftime('%H:%M:%S')}  {msg}", flush=True)


def run_fold(f, iters, complexity, want_test=False):
    tr = np.where(folds != f)[0]
    va = np.where(folds == f)[0]
    t0 = time.time()
    m = CatBoostClassifier(iterations=iters, learning_rate=LR, depth=DEPTH,
                           random_seed=SEED, max_ctr_complexity=complexity,
                           eval_metric="AUC", verbose=0, allow_writing_files=False,
                           thread_count=THREADS)
    m.fit(Pool(L_TR.iloc[tr], y[tr], cat_features=CAT_IDX))
    p = m.predict_proba(Pool(L_TR.iloc[va], cat_features=CAT_IDX))[:, 1]
    p_te = None
    if want_test:
        p_te = m.predict_proba(Pool(L_TE, cat_features=CAT_IDX))[:, 1]
    out = {"va": va, "p": p, "p_te": p_te, "auc": roc_auc_score(y[va], p),
           "secs": time.time() - t0}
    del m
    gc.collect()
    return out


bench = {}
for name, cx in ARMS:
    a = run_fold(PROBE_FOLD, BENCH_EST, cx)
    b = run_fold(PROBE_FOLD, BENCH_EST, cx)
    drift = abs(a["auc"] - b["auc"])
    bench[name] = a["secs"]
    note(f"{name}: bench {a['auc']:.6f} in {hhmm(a['secs'])}, "
         f"repeat drift {drift:.3e} {'OK' if drift == 0.0 else 'DIVERGED'}")

proj = sum(s / BENCH_EST * N_EST * 5 for s in bench.values())
print()
print(f"projected full run: {hhmm(proj)} for {len(ARMS)} arms x 5 folds")
if not SMOKE:
    assert proj < MAX_HOURS * 3600, (
        f"projected {hhmm(proj)} exceeds MAX_HOURS={MAX_HOURS}. "
        "Drop an arm or raise the guard deliberately.")
print("within budget" if proj < MAX_HOURS * 3600 else "SMOKE: guard not enforced")

## Stage 4. The full run

Five folds per arm at the full budget. The test set is predicted by every fold model and
averaged, which is what every other member vector in this repo does.

In [ ]:
results = {}
t_all = time.time()
for name, cx in ARMS:
    oof = np.zeros(len(train))
    test_pred = np.zeros(len(test))
    scores = []
    t0 = time.time()
    for f in range(5):
        r = run_fold(f, N_EST, cx, want_test=True)
        oof[r["va"]] = r["p"]
        test_pred += r["p_te"] / 5
        scores.append(r["auc"])
        done = time.time() - t0
        note(f"  {name} fold {f}: {scores[-1]:.6f}  ({hhmm(r['secs'])}), "
             f"elapsed {hhmm(done)}, about {hhmm(done / (f + 1) * (4 - f))} left")
    cv, sd = float(np.mean(scores)), float(np.std(scores))
    results[name] = {"cv": cv, "sd": sd, "oof": oof, "test": test_pred,
                     "scores": np.array(scores), "cx": cx}
    print(f"{name}: CV {cv:.6f} +/- {sd:.6f}   [{hhmm(time.time() - t0)}]")
    print()

print(f"all arms done in {hhmm(time.time() - t_all)}")

## Stage 5. The paired comparisons

Two of them, and they answer different questions.

**Arm A against row 26** is the experiment: same learner, same folds, same budget, and the
only thing that differs is who computed the target statistics. Row 26's saved out-of-fold
vector is re-scored fold by fold here, so it also has to reproduce its ledger number, which
is the check that the comparison is against what the ledger claims and not against a
drifted artifact.

**Arm B against arm A** is the knob, and is the third independent test of whether pairwise
interaction structure is worth anything on this data.

The bar is the spread of the per-fold differences and how many folds are won, not the fold
spread, which is common to both and cancels. This repo records why, under *Fold spread is
the wrong yardstick for a paired comparison*.

In [ ]:
def paired(a, b, label_a, label_b):
    d = a - b
    print(f"{label_a} vs {label_b}")
    print(f"  per-fold: {np.round(d, 6).tolist()}")
    print(f"  mean {d.mean():+.6f}, paired sd {d.std(ddof=1):.6f}, "
          f"wins {(d > 0).sum()}/5 folds")
    return d


base = np.load(locate(f"{BASELINE_NAME}_oof.npy"))[ROW_IDX]
bf = np.array([roc_auc_score(y[folds == f], base[folds == f]) for f in range(5)])
print(f"row 26 re-scored here: {bf.mean():.6f}, "
      f"{bf.mean() - BASELINE_CV:+.2e} from its ledger number")
print()

A = results["cat_native_c1"]
B = results["cat_native_c2"]
d_a = paired(A["scores"], bf, "arm A (CatBoost's own, 1-way)", "row 26 (our encoder)")
print()
d_b = paired(B["scores"], A["scores"], "arm B (complexity 2)", "arm A (complexity 1)")
print()
print(f"for context, row 38 xgb_te (best single model here) = {XGB_TE_CV:.6f}")
print(f"arm A = {A['cv']:.6f}, arm B = {B['cv']:.6f}")

In [ ]:
pre = "SMOKE_" if SMOKE else ""
for name, r in results.items():
    np.save(OUT / f"{pre}{name}_oof.npy", r["oof"])
    np.save(OUT / f"{pre}{name}_test.npy", r["test"])
    print(f"wrote {pre}{name}_oof.npy, {pre}{name}_test.npy")

print()
print("ledger lines:")
for name, r in results.items():
    print(f"  name    {name}")
    print(f"  cv_mean {r['cv']:.6f}")
    print(f"  cv_std  {r['sd']:.6f}")
print()
print(f"  leak checks {'PASS' if CLEAN else 'FAILED'}, "
      f"fold alignment {'verified' if ALIGNED else 'NOT verified'}")
print()
print("No submission csv. Membership is decided in 36_stack_native.ipynb, which")
print("changes a different variable and gets its own ledger row.")